In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("../data/healthcare-dataset-stroke-data.csv")
df = df.drop(columns="id")
df = df[df["gender"] != "Other"].reset_index(drop=True)
df["bmi_missing"] = df["bmi"].isna().astype(int)

X = df.drop(columns="stroke")
y = df["stroke"]

train/test split

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print("Stroke rate train:", y_train.mean().round(3), "| test:", y_test.mean().round(3))
print("Strokes train:", y_train.sum(), "| test:", y_test.sum())

(4087, 11) (1022, 11)
Stroke rate train: 0.049 | test: 0.049
Strokes train: 199 | test: 50


In [3]:
num_cols = ["age", "avg_glucose_level", "bmi"]
bin_cols = ["hypertension", "heart_disease", "bmi_missing"]
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("bin", "passthrough", bin_cols),
    ("cat", cat_pipe, cat_cols),
])

In [4]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(X_train_prep.shape, X_test_prep.shape)
print(preprocessor.get_feature_names_out())
print("NaN left:", np.isnan(X_train_prep).sum(), np.isnan(X_test_prep).sum())
print("Median learned:", preprocessor.named_transformers_["num"].named_steps["imputer"].statistics_)
print("Train BMI median:", X_train["bmi"].median())

(4087, 18) (1022, 18)
['num__age' 'num__avg_glucose_level' 'num__bmi' 'bin__hypertension'
 'bin__heart_disease' 'bin__bmi_missing' 'cat__gender_Male'
 'cat__ever_married_Yes' 'cat__work_type_Govt_job'
 'cat__work_type_Never_worked' 'cat__work_type_Private'
 'cat__work_type_Self-employed' 'cat__work_type_children'
 'cat__Residence_type_Urban' 'cat__smoking_status_Unknown'
 'cat__smoking_status_formerly smoked' 'cat__smoking_status_never smoked'
 'cat__smoking_status_smokes']
NaN left: 0 0
Median learned: [45.   91.89 28.1 ]
Train BMI median: 28.1


models and cross-validation setup

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

In [10]:
models = {
    "Dummy (always no stroke)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=1000),
    "SVM ": SVC(),
    "Random forest": RandomForestClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "recall": "recall",
    "precision": "precision",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

In [11]:
rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"model": name, **{m: res[f"test_{m}"].mean() for m in scoring}})

results = pd.DataFrame(rows).set_index("model").round(3)
results

C:\Users\windows\OneDrive\Documents\GitHub\stroke-prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\windows\OneDrive\Documents\GitHub\stroke-prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\windows\OneDrive\Documents\GitHub\stroke-prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _

,accuracy,recall,precision,roc_auc,pr_auc
model,,,,,
Dummy (always no stroke),0.951,0.000,0.000,0.500,0.049
Logistic regression,0.950,0.020,0.267,0.853,0.227
SVM,0.951,0.000,0.000,0.671,0.153
Random forest,0.951,0.035,0.486,0.823,0.187


Imbalance and decision threshold

In [12]:
def run_cv(models):
    rows = []
    for name, model in models.items():
        pipe = Pipeline([("prep", preprocessor), ("model", model)])
        res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
        row = {m: res[f"test_{m}"].mean() for m in scoring}
        row["pr_auc_std"] = res["test_pr_auc"].std()
        rows.append({"model": name, **row})
    return pd.DataFrame(rows).set_index("model").round(3)

weighted_models = {
    "Logistic regression (balanced)": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "SVM (balanced)": SVC(class_weight="balanced"),
    "Random forest (balanced)": RandomForestClassifier(class_weight="balanced_subsample", random_state=42),
}
run_cv(weighted_models)

,accuracy,recall,precision,roc_auc,pr_auc,pr_auc_std
model,,,,,,
Logistic regression (balanced),0.755,0.799,0.142,0.854,0.224,0.038
SVM (balanced),0.771,0.644,0.129,0.810,0.173,0.035
Random forest (balanced),0.950,0.020,0.407,0.815,0.175,0.022


In [13]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

best_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

oof_proba = cross_val_predict(best_pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

prec, rec, thr = precision_recall_curve(y_train, oof_proba)
idx = np.where(rec[:-1] >= 0.80)[0][-1]
chosen_thr = thr[idx]
print(f"Threshold: {chosen_thr:.3f} | recall: {rec[idx]:.2f} | precision: {prec[idx]:.2f}")

Threshold: 0.498 | recall: 0.80 | precision: 0.14


Test set evaluation

In [14]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

best_pipe.fit(X_train, y_train)
test_proba = best_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= chosen_thr).astype(int)

print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, digits=3))


[[728 244]
 [ 10  40]]
              precision    recall  f1-score   support

           0      0.986     0.749     0.851       972
           1      0.141     0.800     0.240        50

    accuracy                          0.751      1022
   macro avg      0.564     0.774     0.545      1022
weighted avg      0.945     0.751     0.822      1022



AttributeError: 'float' object has no attribute 'round'

In [15]:
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 3))
print("PR-AUC:", round(average_precision_score(y_test, test_proba), 3))

ROC-AUC: 0.837
PR-AUC: 0.227


coefficients and odds ratios

In [16]:
feature_names = best_pipe.named_steps["prep"].get_feature_names_out()
coefs = best_pipe.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs,
    "odds_ratio": np.exp(coefs),
})
coef_df.reindex(coef_df["coef"].abs().sort_values(ascending=False).index).round(3)

,feature,coef,odds_ratio
0,num__age,1.843,6.318
5,bin__bmi_missing,1.696,5.451
3,bin__hypertension,0.698,2.010
12,cat__work_type_children,0.675,1.964
11,cat__work_type_Self-employed,-0.613,0.542
16,cat__smoking_status_never smoked,-0.345,0.708
14,cat__smoking_status_Unknown,-0.341,0.711
8,cat__work_type_Govt_job,-0.332,0.717
10,cat__work_type_Private,-0.318,0.727
1,num__avg_glucose_level,0.189,1.208


SHAP